# 📊 Capstone 9.2: Data Preparation

**Goal**: Load, explore, and prepare the Wine Quality dataset. Log all findings to MLFlow.

---

In [ ]:
import mlflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
import json
import os

mlflow.autolog(disable=True)
mlflow.set_experiment("capstone-wine-quality")
print("✅ Setup complete!")

In [ ]:
# MLflow Database Configuration
# All modules share a centralized SQLite database at the project root
import os

_db_path = os.path.normpath(
    os.path.join(os.getcwd(), "..", "mlflow.db")
).replace(chr(92), "/")
mlflow.set_tracking_uri(f"sqlite:///{_db_path}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## 1. Load the Dataset

In [ ]:
wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = pd.Series(wine.target, name="target")

# Combine for EDA
df = X.copy()
df["target"] = y
df["target_name"] = df["target"].map(dict(enumerate(wine.target_names)))

print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Classes: {list(wine.target_names)}")
print(f"\nClass distribution:")
print(y.value_counts().sort_index())
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Feature statistics
print("📊 Feature Statistics:")
df.describe().T

In [ ]:
# Distribution of classes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution bar chart
class_counts = y.value_counts().sort_index()
axes[0].bar(wine.target_names, class_counts.values, color=['#3498db', '#2ecc71', '#e74c3c'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Feature correlation heatmap (top features)
corr = X.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='RdBu_r', ax=axes[1],
            vmin=-1, vmax=1, square=True)
axes[1].set_title('Feature Correlations')

plt.tight_layout()
plt.show()

In [ ]:
# Key feature distributions by class
key_features = ['alcohol', 'flavanoids', 'color_intensity', 'proline']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for idx, feature in enumerate(key_features):
    ax = axes[idx // 2, idx % 2]
    for cls_id, cls_name in enumerate(wine.target_names):
        subset = df[df['target'] == cls_id][feature]
        ax.hist(subset, bins=15, alpha=0.5, label=cls_name)
    ax.set_title(f'{feature} by Class')
    ax.legend()

plt.tight_layout()
plt.show()

## 3. Log EDA to MLFlow

In [ ]:
with mlflow.start_run(run_name="data_preparation"):
    mlflow.set_tag("stage", "data_preparation")
    mlflow.set_tag("author", "sujat")
    
    # Dataset info
    mlflow.log_params({
        "dataset": "wine",
        "n_samples": len(X),
        "n_features": X.shape[1],
        "n_classes": len(wine.target_names),
        "test_size": 0.2,
        "random_state": 42,
    })
    
    # Class distribution
    mlflow.log_dict(
        {name: int(count) for name, count in zip(wine.target_names, np.bincount(y))},
        "data/class_distribution.json"
    )
    
    # Feature statistics
    mlflow.log_text(X.describe().to_string(), "data/feature_statistics.txt")
    
    # Save EDA plots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    class_counts = y.value_counts().sort_index()
    axes[0].bar(wine.target_names, class_counts.values, color=['#3498db', '#2ecc71', '#e74c3c'])
    axes[0].set_title('Class Distribution')
    corr = X.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=False, cmap='RdBu_r', ax=axes[1], vmin=-1, vmax=1)
    axes[1].set_title('Feature Correlations')
    plt.tight_layout()
    fig.savefig("eda_overview.png", dpi=150)
    mlflow.log_artifact("eda_overview.png", "plots")
    plt.close()
    os.remove("eda_overview.png")
    
    # Log data sample
    mlflow.log_text(df.head(20).to_csv(index=False), "data/sample_data.csv")
    
    print("✅ Data preparation logged to MLFlow!")

## 4. Prepare Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} samples")
print(f"Test:  {len(X_test)} samples")
print(f"\nTrain class balance: {dict(y_train.value_counts().sort_index())}")
print(f"Test class balance:  {dict(y_test.value_counts().sort_index())}")
print("\n✅ Data ready for training! Proceed to 03_experiment_training.ipynb")